# DO SOFT DRINKS SOLD IN THE UK AND FRANCE CONTAIN LESS SUGAR THAN THOSE SOLD IN GERMANY AND ITALY?

## Introduction

The goal of this project is to check whether soft drinks sold in countries with
a sugar tax contain less sugar than those sold in countries without one. The
idea is that a tax gives manufacturers a financial reason to reformulate their
drinks, so a market where sugary drinks are taxed should end up with less sugar
on the shelves.

**The four countries and why they fall on each side**

| Country | Sugar tax? | Detail |
|---|---|---|
| United Kingdom | Yes | Soft Drinks Industry Levy, in force since April 2018, charged in bands by sugar content |
| France | Yes | Soda tax since 2012, restructured in 2018 so the rate scales with sugar content |
| Germany | No | No tax; relies on a voluntary industry reduction agreement instead |
| Italy | Legislated, never in force | Enacted in the 2020 Budget Law (Law 160/2019) but postponed repeatedly and still not applied |

Italy is the case that needs care. A sugar tax exists in Italian law, but its
entry into force has been deferred again and again and it has never actually
been applied, most recently pushed back to January 2027. For this project Italy
therefore counts as an untaxed market: no Italian manufacturer has yet faced a
financial incentive to reformulate, which is what the comparison is testing.

The UK and France both tax sugary soft drinks, while Germany and Italy do not,
so comparing the sugar content of sodas across these four countries should show
whether the policy leaves a mark on the products themselves.

**Policy background**

*United Kingdom — sugar tax*
- HMRC, "Check if your drink is liable for the Soft Drinks Industry Levy."
  Primary source for the levy's thresholds: liable drinks are those with at
  least 5g of sugar per 100ml, with a higher band at 8g per 100ml.
  https://www.gov.uk/guidance/check-if-your-drink-is-liable-for-the-soft-drinks-industry-levy
- GOV.UK, "Soft Drinks Industry Levy: detailed information." Collection page
  covering registration, returns and rates.
  https://www.gov.uk/government/collections/soft-drinks-industry-levy-detailed-information

*France — sugar tax*
- Sénat, "La fiscalité comportementale en santé : stop ou encore ?" Parliamentary
  report describing the 2012 flat-rate contribution on sweetened drinks and its
  2018 restructuring into a scale that rises with added sugar per hectolitre.
  https://www.senat.fr/rap/r23-638/r23-63811.html

*Germany — no tax*
- von Philipsborn et al., "Interim Evaluation of Germany's Sugar Reduction
  Strategy for Soft Drinks," Annals of Nutrition and Metabolism. Peer-reviewed
  evaluation confirming Germany relies on voluntary industry commitments rather
  than a tax, and measuring how far short of target they fell.
  https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10568594/
- BMEL, "Nationale Reduktions- und Innovationsstrategie für Zucker, Fette und
  Salz in Fertigprodukten" (2018). The German government strategy itself.
  https://www.bmel.de/SharedDocs/Downloads/Ernaehrung/NationaleReduktionsInnovationsstrategie-Layout.pdf

*Italy — legislated but never in force*
- PwC Worldwide Tax Summaries, "Italy — Significant developments." Records the
  most recent postponement of the sugar tax's effective date.
  https://taxsummaries.pwc.com/italy/corporate/significant-developments
- PwC TLS Italy, "Sugar Tax – Further postponement approved by the Italian
  Government" (June 2025). Describes the Council of Ministers decision and the
  scope of the tax under Law 160/2019.
  https://blog.pwc-tls.it/en/2025/06/23/sugar-tax-further-postponement-approved-by-the-italian-government/

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

In [ ]:
# Used capital letters to clarify that these variables will not be changed throughout the project. Fixed Variables. 
URL = "https://search.openfoodfacts.org/search"
HEADERS = {"User-Agent": "ME204-LSE-project/1.0 j.milagro-caro@lse.ac.uk"}

## The API

The data for this project comes from the Open Food Facts API. Open Food Facts is a free, open database of food products from around the world, built by volunteers who scan barcodes and upload the information from the packaging. Each product in the database has a barcode, a name, a brand, a list of countries where it is sold, and a set of nutritional values taken from the label. The API is free to use and does not need a key or an account for reading data, which is why it works well for a project like this one.

### How I found the right endpoint

- I started from the API documentation homepage, which lists the different versions of the API and explains what each one can do.
- The documentation has a table comparing versions, and it shows that structured search (filtering by things like category and country) is only available in version 2, at `/api/v2/search`. Version 3 is newer but cannot search.
- The reference pages showed me the names of the filters I needed: `categories_tags_en` to pick a food category, and `countries_tags_en` to pick a country. They also explained `fields`, which lets me ask for only the columns I need, and `page_size`, which controls how many products come back at once.
- The Authentication section asks every user to send a custom User-Agent header with an app name and a contact email, so I added that to my requests.
- Every request I made to this endpoint returned a 503 error. I tested the same request in a browser and confirmed it was correctly formed, which ruled out a mistake in my own code, and the error page said the service was under heavy load and deprioritising anonymous users.
- Searching for the problem showed that other developers had reported the same error on this endpoint earlier in the year, and that the older search backend is being retired in favour of a newer service called Search-a-licious, hosted at `search.openfoodfacts.org`. This service is mentioned in the official documentation, but only as the future home of full-text search, not as a replacement for the endpoint I was using.
- I opened the documentation for the new service, which lists a `GET /search` endpoint, and tested it in a browser before changing any code. It returned both products and nutritional values, so I rewrote my collection function to use it.
- The new endpoint asks the question differently. Instead of separate `categories_tags_en` and `countries_tags_en` filters, it takes a single `q` parameter containing both conditions, written as `categories_tags:"en:sodas" AND countries_tags:"en:italy"`. It also returns the products under a key called `hits` rather than `products`.
- The structure of my pipeline did not change. Only the URL and the way the filters are written were different, and everything after the request stayed the same.

### Limitations of the API

- The endpoint the documentation recommends for structured search, `/api/v2/search`, was returning errors throughout the time I was collecting data. Depending on a service that can refuse requests is a real limitation. My collection function does not retry automatically: it reports the status code and moves on, and I re-ran the cell manually when a request failed. A more robust pipeline would retry failed requests after a delay.
- The service I ended up using, Search-a-licious, is still in beta. It works, but it may change, and the official documentation has not yet been updated to describe it as the replacement for the older search endpoint.
- The main API is rate limited, and requests from anonymous users are deprioritised when the servers are busy. The limits for the newer search service are not documented in the same place, so I do not know the exact ceiling. With only four requests to make, one per country, I stayed well below any plausible limit, but a larger collection would need a deliberate delay between requests.
- Each request returns a limited number of products. I requested a single page of up to 100 products per country rather than paginating through the full category, which caps my sample and is a limitation I return to below.
- Filters have to match the exact tag names used in the Open Food Facts taxonomy. A misspelled tag returns an empty result rather than an error message, so it is easy to think a category is empty when the tag is simply wrong.
- The data is added by volunteers, and the documentation itself warns that there is no guarantee it is accurate or complete. Many products are missing nutritional values entirely.
- The documentation asks users not to pull more than a few hundred products through the API, and to download the full dataset as a file instead. This puts a practical ceiling on how much data I can collect this way.

In [ ]:
RAW_DIR = Path("../data/raw")

# Store the countries we will be using in the API as a list
COUNTRIES = ["united-kingdom", "france", "germany", "italy"]

In [ ]:
def collect_sodas(country, data_folder=RAW_DIR):
    """Fetch sodas for one country and save the raw JSON."""
    data_folder = Path(data_folder)
    data_folder.mkdir(parents=True, exist_ok=True)

    params = {
        "q": f'categories_tags:"en:sodas" AND countries_tags:"en:{country}"',
        "fields": "code,product_name,brands,nutriments",
        "page_size": 100,
    }

    response = requests.get(URL, params=params, headers=HEADERS, timeout=60)

    if response.status_code != 200:
        return f"Error: {response.status_code} for {country}"

    path = data_folder / f"{country}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)

    return "success"

for country in COUNTRIES: 
    print(country, collect_sodas(country))

united-kingdom success
france success
germany success
italy success


In [ ]:
# This cell of code helps me identify how the JSON is structred in order to create my DataFrame later on. 

with open(RAW_DIR / "france.json", encoding="utf-8") as f:
    sample = json.load(f)

print(json.dumps(sample["hits"][0], indent=2))

{
  "code": "5449000285720",
  "brands": [
    "Fanta"
  ],
  "nutriments": {
    "energy-kcal_100g": 5,
    "carbohydrates_100g": 0,
    "proteins_100g": 0,
    "sugars_100g": 0,
    "saturated-fat_100g": 0,
    "fat_100g": 0
  },
  "product_name": "Fanta Rasberry"
}


## The shape of the data

The response is a dictionary. The products sit under `hits`, and each product has:

- `code` — the barcode, a unique identifier
- `product_name` and `brands` — for identifying the drink
- `nutriments` — a nested dictionary, where `sugars_100g` is the field this
  project depends on

In NB02 I flatten `nutriments` out of the nested structure so each product
becomes one row.

**One thing the sample product already shows.** The first French product is
"Fanta Rasberry" with `sugars_100g: 0` — a zero-sugar variant. The category
mixes full-sugar and diet drinks, and that matters for my question: a sugar tax
pushes manufacturers both to reformulate existing drinks and to launch more diet
lines. If taxed countries simply carry more zero-sugar variants, the average
sugar figure falls for a reason that is not reformulation. I return to how I
handle this in NB02.

In [ ]:
# Check that I indeed have 100 entries per country. 
for country in COUNTRIES:
    with open(RAW_DIR / f"{country}.json", encoding="utf-8") as f:
        result = json.load(f)
    print(f"{country}: {len(result['hits'])} retrieved, {result.get('count')} total matches")

united-kingdom: 100 retrieved, 512 total matches
france: 100 retrieved, 3765 total matches
germany: 100 retrieved, 1943 total matches
italy: 100 retrieved, 337 total matches


## Checking what I collected

Before using the data I check how many products each request actually returned,
and how many exist in the database in total.

- **Retrieved** is how many products I have. The ceiling is my `page_size` of 100.
- **Total matches** is how many the database holds for that country and category.

I now know I have 100 samples retrieved per country. 

## Sources and tools

**Data source**

- Open Food Facts, a free and open database of food products built by volunteers. The data is available under the Open Database License. https://world.openfoodfacts.org

**API documentation**

- Introduction to Open Food Facts API documentation. Used to understand what the API offers, which version supports structured search, the rate limits, and the requirement to send a custom User-Agent header. https://openfoodfacts.github.io/openfoodfacts-server/api/
- Reference OpenAPI documentation for API v2. Used to find the names of the search filters, and the `fields` and `page_size` parameters. https://openfoodfacts.github.io/openfoodfacts-server/api/ref-v2/
- Tutorial on using the Open Food Facts API. Used to see worked examples of search requests. https://openfoodfacts.github.io/openfoodfacts-server/api/tutorial-off-api/
- Reference: API Cheatsheet. Used to check common request patterns. https://openfoodfacts.github.io/openfoodfacts-server/api/ref-cheatsheet/
- Search-a-licious API documentation. The interactive documentation for the newer search service, used to find the `GET /search` endpoint and its parameters after the older endpoint stopped responding. https://search.openfoodfacts.org/docs
- Terms of use and reuse. Read before collecting any data. https://world.openfoodfacts.org/terms-of-use

**Course materials**

- ME204 lecture and lab notebooks from weeks 1 and 2, in particular W01D03 on fetching data from an API with `requests` and saving JSON, W01D04 and W02D01 on building and grouping dataframes, and W02D04 on writing a collection function, looping over a list of items, and saving one file per item. The structure of my collection function follows the pattern taught in the W02D04 lab.

**AI tools**

- Claude was used throughout the data collection process as a source of guidance. Its main contribution was helping me diagnose why every request to the documented search endpoint returned a 503 error. It suggested testing the same request in a browser to establish whether the problem was in my own code, and when that showed the request was correctly formed, it searched for reports of the same issue and found that the older search backend was being retired in favour of Search-a-licious. It then pointed me to the documentation for that service, which I read and tested myself before rewriting my collection function.
- Claude was also used to find sources for the policy claims behind my research
question. My introduction rests on the UK and France taxing sugary soft drinks
while Germany and Italy do not, and I had originally written this without citing
anything. I asked Claude to direct me to realiable sources that explain the taxing situation for each country.